# Synchronize MCP and A2A Server Metadata with Registry

Auto-populate registry records by providing an MCP server or A2A agent URL. The system connects to the endpoint, fetches metadata, and populates the record automatically.

![image](image/image.png)

## What You'll Learn

1. Sync a public MCP server (no auth)
2. Sync an A2A agent card
3. Sync an OAuth-protected MCP server (Cognito + AgentCore Gateway)
4. Update and re-sync a record
5. Handle failure cases
6. Full governance workflow — Publisher → Approver → Consumer

## Prerequisites

- Updated CLI models with sync support (see `URL_SYNC_GUIDE.md`)
- IAM permissions including `*WorkloadIdentity`, `GetWorkloadAccessToken`, `GetResourceOauth2Token` (see `IAM_PERMISSIONS.md`)
- **New registry required** — old registries lack the workload identity needed for OAuth/IAM sync

In [ ]:
!pip install -q "boto3>=1.42.87"

In [ ]:
import boto3
import json
import time
import os

# Configuration
AWS_REGION = "us-west-2"
AWS_PROFILE = ""  # Change to your profile

session = boto3.Session(profile_name=AWS_PROFILE, region_name=AWS_REGION)
ac = session.client("bedrock-agentcore-control")
iam = session.client("iam")
cognito = session.client("cognito-idp")
sts = session.client("sts")

ACCOUNT_ID = sts.get_caller_identity()["Account"]

def pp(resp):
    """Pretty-print API response."""
    print(json.dumps({k: v for k, v in resp.items() if k != "ResponseMetadata"}, indent=2, default=str))

def wait_record(registry_id, record_id, timeout=60):
    """Poll until record reaches a terminal status."""
    for _ in range(timeout // 5):
        r = cp.get_registry_record(registryId=registry_id, recordId=record_id)
        status = r["status"]
        print(f"  status: {status}")
        if status in ("DRAFT", "CREATE_FAILED", "APPROVED"):
            return r
        time.sleep(5)
    return r

print(f"Account: {ACCOUNT_ID} | Region: {AWS_REGION}")

---
## 1. Create a New Registry

A new registry is required — it gets a workload identity that enables OAuth/IAM sync.

In [ ]:
from utils import (
    create_registry, seed, wait_for_search_index,
    search, delete_registry, get_cp_client, get_dp_client, REGION,
)

registry = create_registry(name="URLSyncTest", description="URL sync testing")
REGISTRY_ID = registry["registryArn"].split("/")[-1]
print(f"Registry ID: {REGISTRY_ID}")

In [ ]:
REGISTRY_ID = registry["registryId"]
REGISTRY_ARN = registry["registryArn"]
print(f"Registry ID: {REGISTRY_ID}")

---
## 2. Sync a Public MCP Server (No Auth)

The simplest case — just provide a URL.

In [ ]:
cp = get_cp_client()
r = cp.create_registry_record(
    registryId=REGISTRY_ID,
    name="mcp_no_auth",
    descriptorType="MCP",
    synchronizationType="URL",
    synchronizationConfiguration={
        "fromUrl": {"url": "https://knowledge-mcp.global.api.aws"}
    },
)
RECORD_MCP = r["recordArn"].split("/")[-1]
print(f"Record: {RECORD_MCP} | Status: {r['status']}")

# Wait for sync to complete
for _ in range(24):
    record = cp.get_registry_record(registryId=REGISTRY_ID, recordId=RECORD_MCP)
    status = record.get("status", "UNKNOWN")
    print(f"  status: {status}")
    if status in ("DRAFT", "APPROVED", "CREATE_FAILED"):
        break
    time.sleep(5)

print(f"\nName: {record['name']}")
print(f"Status: {record['status']}")

if record["status"] == "CREATE_FAILED":
    print(f"Reason: {record.get('statusReason', 'unknown')}")
    print("⚠️ Rate limited — delete this record and retry in 60 seconds:")
    print(f"  cp.delete_registry_record(registryId=REGISTRY_ID, recordId='{RECORD_MCP}')")
else:
    tools = json.loads(record["descriptors"]["mcp"]["tools"]["inlineContent"])
    print(f"Tools synced: {len(tools['tools'])}")
    for t in tools["tools"]:
        print(f"  - {t['name']}: {t.get('description', '')[:80]}")


---
## 3. Sync an A2A Agent Card

In [ ]:
cp = get_cp_client()
r = cp.create_registry_record(
    registryId=REGISTRY_ID,
    name="a2a_agent",
    descriptorType="A2A",
    synchronizationType="URL",
    synchronizationConfiguration={
        "fromUrl": {"url": "https://agent.willform.ai/.well-known/agent-card.json"}
    },
)
RECORD_A2A = r["recordArn"].split("/")[-1]
print(f"Record: {RECORD_A2A} | Status: {r['status']}")

record = wait_record(REGISTRY_ID, RECORD_A2A)
print(f"\nName: {record['name']} | Status: {record['status']}")

---
## 4. Sync an OAuth-Protected MCP Server

This sets up a Cognito user pool, an AgentCore Gateway with Cognito JWT auth, and an OAuth2 credential provider. Then syncs from the protected gateway.

In [ ]:
# 4a. Create Cognito user pool + M2M client
pool = cognito.create_user_pool(PoolName="url-sync-test-pool")
POOL_ID = pool["UserPool"]["Id"]
COGNITO_DOMAIN = f"url-sync-test-{ACCOUNT_ID[:8]}"

cognito.create_user_pool_domain(UserPoolId=POOL_ID, Domain=COGNITO_DOMAIN)
cognito.create_resource_server(
    UserPoolId=POOL_ID, Identifier="mcp-gateway", Name="MCP Gateway",
    Scopes=[{"ScopeName": "invoke", "ScopeDescription": "Invoke MCP"}],
)

client = cognito.create_user_pool_client(
    UserPoolId=POOL_ID, ClientName="sync-m2m", GenerateSecret=True,
    AllowedOAuthFlows=["client_credentials"],
    AllowedOAuthScopes=["mcp-gateway/invoke"],
    AllowedOAuthFlowsUserPoolClient=True,
)
CLIENT_ID = client["UserPoolClient"]["ClientId"]
CLIENT_SECRET = client["UserPoolClient"]["ClientSecret"]
print(f"Cognito Pool: {POOL_ID} | Client: {CLIENT_ID}")

In [ ]:
# 4b. Create gateway with Cognito JWT auth
gw = ac.create_gateway(
    name="url-sync-test-gw", protocolType="MCP", authorizerType="CUSTOM_JWT",
    authorizerConfiguration={"customJWTAuthorizer": {
        "discoveryUrl": f"https://cognito-idp.{AWS_REGION}.amazonaws.com/{POOL_ID}/.well-known/openid-configuration",
        "allowedClients": [CLIENT_ID],
    }},
    roleArn=f"arn:aws:iam::{ACCOUNT_ID}:role/AgentCoreGatewayExecutionRole",
)
GW_ID = gw["gatewayId"]
print(f"Gateway: {GW_ID} — waiting for READY...")

for _ in range(12):
    s = ac.get_gateway(gatewayIdentifier=GW_ID)["status"]
    if s == "READY": break
    time.sleep(5)

ac.create_gateway_target(
    gatewayIdentifier=GW_ID, name="knowledge-mcp",
    targetConfiguration={"mcp": {"mcpServer": {"endpoint": "https://knowledge-mcp.global.api.aws"}}},
)
MCP_OAUTH_URL = ac.get_gateway(gatewayIdentifier=GW_ID)["gatewayUrl"]
print(f"Gateway URL: {MCP_OAUTH_URL}")

In [ ]:
# 4c. Create OAuth2 credential provider
TOKEN_EP = f"https://{COGNITO_DOMAIN}.auth.{AWS_REGION}.amazoncognito.com/oauth2/token"
AUTH_EP = f"https://{COGNITO_DOMAIN}.auth.{AWS_REGION}.amazoncognito.com/oauth2/authorize"
ISSUER = f"https://cognito-idp.{AWS_REGION}.amazonaws.com/{POOL_ID}"

provider = ac.create_oauth2_credential_provider(
    name="url-sync-test-cognito",
    credentialProviderVendor="CognitoOauth2",
    oauth2ProviderConfigInput={"includedOauth2ProviderConfig": {
        "clientId": CLIENT_ID, "clientSecret": CLIENT_SECRET,
        "tokenEndpoint": TOKEN_EP, "issuer": ISSUER, "authorizationEndpoint": AUTH_EP,
    }},
)
OAUTH_ARN = provider["credentialProviderArn"]
print(f"OAuth Provider: {OAUTH_ARN}")

In [ ]:
# 4d. Sync from OAuth-protected gateway
r = cp.create_registry_record(
    registryId=REGISTRY_ID,
    name="mcp_oauth",
    descriptorType="MCP",
    synchronizationType="URL",
    synchronizationConfiguration={
        "fromUrl": {
            "url": MCP_OAUTH_URL,
            "credentialProviderConfigurations": [{
                "credentialProviderType": "OAUTH",
                "credentialProvider": {
                    "oauthCredentialProvider": {
                        "providerArn": OAUTH_ARN,
                        "grantType": "CLIENT_CREDENTIALS",
                        "scopes": ["mcp-gateway/invoke"],
                    }
                },
            }],
        }
    },
)
RECORD_OAUTH = r["recordArn"].split("/")[-1]
print(f"Record: {RECORD_OAUTH} | Status: {r['status']}")

record = wait_record(REGISTRY_ID, RECORD_OAUTH)
print(f"\nName: {record['name']} | Status: {record['status']}")
if record.get("statusReason"):
    print(f"Reason: {record['statusReason']}")

---
## 5. Update and Re-sync

Change the sync URL on an existing record and trigger re-sync.

In [ ]:
cp.update_registry_record(
    registryId=REGISTRY_ID,
    recordId=RECORD_MCP,
    synchronizationConfiguration={
        "optionalValue": {"fromUrl": {"url": "https://knowledge-mcp.global.api.aws"}}
    },
    triggerSynchronization=True,
)
print("Re-sync triggered")
record = wait_record(REGISTRY_ID, RECORD_MCP)
print(f"Name: {record['name']} | Status: {record['status']}")

---
## 6. Failure Case — Bad URL

In [ ]:
r = cp.create_registry_record(
    registryId=REGISTRY_ID,
    name="bad_url",
    descriptorType="MCP",
    synchronizationType="URL",
    synchronizationConfiguration={
        "fromUrl": {"url": "https://nonexistent.example.com/mcp"}
    },
)
RECORD_BAD = r["recordArn"].split("/")[-1]
record = wait_record(REGISTRY_ID, RECORD_BAD)
print(f"\nStatus: {record['status']}")
print(f"Reason: {record.get('statusReason', 'N/A')}")

---
## 7. Registry Record Lifecycle

Walk through the full lifecycle of a registry record — from creation via URL sync, through approval, to consumer discovery. Each step uses a different persona (Publisher, Approver, Consumer) to illustrate the API calls involved.

### 7a. Publisher — Create and Submit for Approval

In [ ]:
# Publisher: create a record via URL sync
r = cp.create_registry_record(
    registryId=REGISTRY_ID,
    name="publisher_mcp_record",
    descriptorType="MCP",
    synchronizationType="URL",
    synchronizationConfiguration={
        "fromUrl": {"url": "https://knowledge-mcp.global.api.aws"}
    },
)
RECORD_GOV = r["recordArn"].split("/")[-1]
record = wait_record(REGISTRY_ID, RECORD_GOV)
print(f"Publisher created: {record['name']} | Status: {record['status']}")

# Publisher: submit for approval
cp.submit_registry_record_for_approval(
    registryId=REGISTRY_ID, recordId=RECORD_GOV
)
record = cp.get_registry_record(registryId=REGISTRY_ID, recordId=RECORD_GOV)
print(f"Publisher submitted → Status: {record['status']}")

### 7b. Approver — Review and Approve

In [ ]:
# Approver: review the record
record = cp.get_registry_record(registryId=REGISTRY_ID, recordId=RECORD_GOV)
print(f"Approver reviewing: {record['name']}")
print(f"  Status: {record['status']}")
print(f"  Type: {record['descriptorType']}")

# Show tools that were auto-populated
tools = json.loads(record["descriptors"]["mcp"]["tools"]["inlineContent"])
print(f"  Tools ({len(tools['tools'])}):\n")
for t in tools["tools"]:
    print(f"    - {t['name']}: {t.get('description', '')[:60]}")

# Approver: approve the record
cp.update_registry_record_status(
    registryId=REGISTRY_ID,
    recordId=RECORD_GOV,
    status="APPROVED",
)
record = cp.get_registry_record(registryId=REGISTRY_ID, recordId=RECORD_GOV)
print(f"\nApprover approved → Status: {record['status']}")

### 7c. Consumer — Discover via Search

In [ ]:
# Consumer: search for approved records
dp = session.client(
    "bedrock-agentcore",
)

results = dp.search_registry_records(
    registryIds=[REGISTRY_ID],
    searchQuery="documentation",
)

print(f"Consumer search results ({len(results.get('registryRecords', []))}):\n")
for rec in results.get("registryRecords", []):
    print(f"  - {rec['name']} [{rec['descriptorType']}] — {rec['status']}")
    print(f"    {rec.get('description', 'No description')[:80]}")

### 7d. Publisher — Update Approved Record (triggers re-review)

When a publisher updates an approved record and re-syncs, it moves back to `DRAFT`. The approved revision remains searchable until the new version is approved.

In [ ]:
# Publisher: update and re-sync an approved record
cp.update_registry_record(
    registryId=REGISTRY_ID,
    recordId=RECORD_GOV,
    synchronizationConfiguration={
        "optionalValue": {"fromUrl": {"url": "https://knowledge-mcp.global.api.aws"}}
    },
    triggerSynchronization=True,
)
record = wait_record(REGISTRY_ID, RECORD_GOV)
print(f"\nPublisher re-synced → Status: {record['status']}")
print("(Approved revision still searchable by consumers until this version is re-approved)")

---
## 8. Cleanup

Delete all test resources.

In [ ]:
# Delete records
records = cp.list_registry_records(registryId=REGISTRY_ID)
for rec in records.get("registryRecords", []):
    rid = rec["recordId"]
    print(f"Deleting record {rid}...")
    cp.delete_registry_record(registryId=REGISTRY_ID, recordId=rid)

time.sleep(10)

# Delete registry
print("Deleting registry...")
cp.delete_registry(registryId=REGISTRY_ID)

# Delete gateway targets + gateway
try:
    targets = ac.list_gateway_targets(gatewayIdentifier=GW_ID)
    for t in targets.get("items", []):
        ac.delete_gateway_target(gatewayIdentifier=GW_ID, targetId=t["targetId"])
    time.sleep(5)
    ac.delete_gateway(gatewayIdentifier=GW_ID)
    print(f"Deleted gateway {GW_ID}")
except: pass

# Delete OAuth provider
try:
    ac.delete_oauth2_credential_provider(name="url-sync-test-cognito")
    print("Deleted OAuth provider")
except: pass

# Delete Cognito
try:
    cognito.delete_user_pool_domain(UserPoolId=POOL_ID, Domain=COGNITO_DOMAIN)
    cognito.delete_user_pool(UserPoolId=POOL_ID)
    print(f"Deleted Cognito pool {POOL_ID}")
except: pass

print("\nCleanup complete!")